STAGE 4.9
# FINAL OPENVINO DEPLOYMENT EVALUATION & EXPERIMENT CONSOLIDATION

In [1]:
# =============================================================================
# CISLR — STAGE 4.9
# FINAL OPENVINO DEPLOYMENT EVALUATION & EXPERIMENT CONSOLIDATION
#
# PURPOSE:
#   Consolidate all validated evidence from Stages 4.3–4.8 into a single
#   final deployment evaluation.
#
# IMPORTANT:
#   - NO retraining
#   - NO OpenVINO reconversion
#   - NO CISLR video decoding
#   - NO official retrieval rerun
#   - NO Stage 4.3 / 4.4 / 4.5 / 4.6 / 4.7 / 4.8 rerun
#
# INPUT STAGES:
#   4.3 Computational Benchmark
#   4.4 Model Training
#   4.5 Official CISLR Evaluation
#   4.6 Model Comparison & Selection
#   4.7 OpenVINO Conversion
#   4.8 OpenVINO Benchmark
#
# OUTPUT:
#   One final evidence package for the CISLR OpenVINO project.
# =============================================================================

import os
import sys
import json
import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

# -------------------------------------------------------------------------
# Stage 4.3
# -------------------------------------------------------------------------

STAGE_4_3_DIR = (
    ROOT /
    "audit" /
    "stage4_3_benchmark"
)

STAGE_4_3_CSV = (
    STAGE_4_3_DIR /
    "stage4_3_computational_benchmark.csv"
)

STAGE_4_3_JSON = (
    STAGE_4_3_DIR /
    "stage4_3_computational_benchmark.json"
)

# -------------------------------------------------------------------------
# Stage 4.4
# -------------------------------------------------------------------------

STAGE_4_4_DIR = (
    ROOT /
    "audit" /
    "stage4_4_training"
)

# -------------------------------------------------------------------------
# Stage 4.5
# -------------------------------------------------------------------------

STAGE_4_5_DIR = (
    ROOT /
    "audit" /
    "stage4_5_official_evaluation"
)

STAGE_4_5_COMPARISON = (
    STAGE_4_5_DIR /
    "stage4_5_official_model_comparison.csv"
)

STAGE_4_5_COMPARISON_JSON = (
    STAGE_4_5_DIR /
    "stage4_5_official_model_comparison.json"
)

# -------------------------------------------------------------------------
# Stage 4.6
# -------------------------------------------------------------------------

STAGE_4_6_DIR = (
    ROOT /
    "audit" /
    "stage4_6_model_comparison"
)

# -------------------------------------------------------------------------
# Stage 4.7
# -------------------------------------------------------------------------

STAGE_4_7_DIR = (
    ROOT /
    "audit" /
    "stage4_7_openvino_conversion"
)

STAGE_4_7_XML = (
    STAGE_4_7_DIR /
    "fp32" /
    "mobilenet_v3_large_stage4_7_fp32.xml"
)

STAGE_4_7_BIN = (
    STAGE_4_7_DIR /
    "fp32" /
    "mobilenet_v3_large_stage4_7_fp32.bin"
)

STAGE_4_7_VALIDATION = (
    STAGE_4_7_DIR /
    "stage4_7_conversion_validation.csv"
)

STAGE_4_7_METADATA = (
    STAGE_4_7_DIR /
    "stage4_7_conversion_metadata.json"
)

# -------------------------------------------------------------------------
# Stage 4.8
# -------------------------------------------------------------------------

STAGE_4_8_DIR = (
    ROOT /
    "audit" /
    "stage4_8_openvino_benchmark"
)

STAGE_4_8_COMPARISON = (
    STAGE_4_8_DIR /
    "stage4_8_pytorch_vs_openvino_comparison.csv"
)

STAGE_4_8_BENCHMARK = (
    STAGE_4_8_DIR /
    "stage4_8_openvino_benchmark.json"
)

STAGE_4_8_LATENCIES = (
    STAGE_4_8_DIR /
    "stage4_8_latency_measurements.csv"
)

# -------------------------------------------------------------------------
# Final Stage 4.9 output
# -------------------------------------------------------------------------

OUTPUT_DIR = (
    ROOT /
    "audit" /
    "stage4_9_final_deployment_evaluation"
)

PLOTS_DIR = (
    OUTPUT_DIR /
    "plots"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PLOTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. HEADER
# =============================================================================

print("=" * 80)
print("CISLR — STAGE 4.9 FINAL OPENVINO DEPLOYMENT EVALUATION")
print("=" * 80)

print()
print("Project root:")
print(ROOT)

print()
print("Final selected model:")
print("MobileNetV3-Large")

print()
print("Final deployment backend:")
print("OpenVINO FP32")

print()
print("Output:")
print(OUTPUT_DIR)


# =============================================================================
# 3. ENVIRONMENT
# =============================================================================

print()
print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print("Python       :", sys.version)
print("Platform     :", platform.platform())
print("Architecture :", platform.machine())


# =============================================================================
# 4. FILE VERIFICATION
# =============================================================================

print()
print("=" * 80)
print("VERIFYING PREVIOUS STAGE ARTIFACTS")
print("=" * 80)


required_files = {
    "Stage 4.3 computational CSV":
        STAGE_4_3_CSV,

    "Stage 4.3 computational JSON":
        STAGE_4_3_JSON,

    "Stage 4.5 retrieval comparison":
        STAGE_4_5_COMPARISON,

    "Stage 4.5 retrieval comparison JSON":
        STAGE_4_5_COMPARISON_JSON,

    "Stage 4.7 OpenVINO XML":
        STAGE_4_7_XML,

    "Stage 4.7 OpenVINO BIN":
        STAGE_4_7_BIN,

    "Stage 4.7 validation CSV":
        STAGE_4_7_VALIDATION,

    "Stage 4.7 metadata":
        STAGE_4_7_METADATA,

    "Stage 4.8 comparison CSV":
        STAGE_4_8_COMPARISON,

    "Stage 4.8 benchmark JSON":
        STAGE_4_8_BENCHMARK,

    "Stage 4.8 latency CSV":
        STAGE_4_8_LATENCIES
}


missing_files = []

for description, path in required_files.items():

    exists = path.exists()

    print(
        f"{description:<40} : "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:
        missing_files.append(
            (description, path)
        )


if missing_files:

    print()
    print("Missing required artifacts:")

    for description, path in missing_files:
        print(
            f"  - {description}: {path}"
        )

    raise FileNotFoundError(
        "Required previous-stage artifact(s) are missing."
    )


print()
print("All required artifacts found.")


# =============================================================================
# 5. HELPER FUNCTIONS
# =============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def load_json(path):

    with open(path, "r") as f:
        return json.load(f)


# =============================================================================
# 6. LOAD STAGE 4.3 COMPUTATIONAL RESULTS
# =============================================================================

print()
print("=" * 80)
print("LOADING STAGE 4.3 COMPUTATIONAL BENCHMARK")
print("=" * 80)

compute_df = pd.read_csv(
    STAGE_4_3_CSV
)

compute_json = load_json(
    STAGE_4_3_JSON
)

print(
    "Rows:",
    len(compute_df)
)

print(
    "Columns:",
    list(compute_df.columns)
)


# =============================================================================
# 7. LOAD STAGE 4.5 OFFICIAL RETRIEVAL RESULTS
# =============================================================================

print()
print("=" * 80)
print("LOADING STAGE 4.5 OFFICIAL CISLR EVALUATION")
print("=" * 80)

retrieval_df = pd.read_csv(
    STAGE_4_5_COMPARISON
)

retrieval_json = load_json(
    STAGE_4_5_COMPARISON_JSON
)

print(
    "Rows:",
    len(retrieval_df)
)

print(
    "Columns:",
    list(retrieval_df.columns)
)

print()
print("Official retrieval comparison:")
print(
    retrieval_df.to_string(
        index=False
    )
)


# =============================================================================
# 8. LOAD STAGE 4.7 CONVERSION VALIDATION
# =============================================================================

print()
print("=" * 80)
print("LOADING STAGE 4.7 OPENVINO VALIDATION")
print("=" * 80)

conversion_df = pd.read_csv(
    STAGE_4_7_VALIDATION
)

conversion_metadata = load_json(
    STAGE_4_7_METADATA
)

print(
    "Rows:",
    len(conversion_df)
)

print(
    conversion_df.to_string(
        index=False
    )
)


# =============================================================================
# 9. LOAD STAGE 4.8 BENCHMARK
# =============================================================================

print()
print("=" * 80)
print("LOADING STAGE 4.8 OPENVINO BENCHMARK")
print("=" * 80)

openvino_comparison_df = pd.read_csv(
    STAGE_4_8_COMPARISON
)

openvino_benchmark = load_json(
    STAGE_4_8_BENCHMARK
)

latency_df = pd.read_csv(
    STAGE_4_8_LATENCIES
)

print()
print("PyTorch vs OpenVINO:")
print(
    openvino_comparison_df.to_string(
        index=False
    )
)

print()
print("Latency measurements:")
print(
    latency_df.groupby("device")[
        "latency_ms"
    ].agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
            "std"
        ]
    ).to_string()
)


# =============================================================================
# 10. EXTRACT MOBILE NET RETRIEVAL RESULTS
# =============================================================================

print()
print("=" * 80)
print("EXTRACTING FINAL MOBILE NET RESULTS")
print("=" * 80)


def find_model_row(df, model_name):

    possible_columns = [
        "model",
        "Model",
        "architecture",
        "Architecture",
        "model_name",
        "Model Name"
    ]

    for column in possible_columns:

        if column in df.columns:

            mask = (
                df[column]
                .astype(str)
                .str.lower()
                .str.contains(
                    model_name.lower()
                )
            )

            matches = df[mask]

            if len(matches) > 0:
                return matches.iloc[0]

    raise ValueError(
        f"Could not locate {model_name} in dataframe."
    )


mobile_retrieval = find_model_row(
    retrieval_df,
    "MobileNetV3-Large"
)

print()
print("MobileNetV3-Large official retrieval:")
print(
    mobile_retrieval.to_string()
)


# =============================================================================
# 11. EXTRACT STAGE 4.3 MOBILE NET COMPUTE RESULTS
# =============================================================================

mobile_compute = find_model_row(
    compute_df,
    "MobileNetV3-Large"
)

print()
print("MobileNetV3-Large Stage 4.3 compute:")
print(
    mobile_compute.to_string()
)


# =============================================================================
# 12. EXTRACT STAGE 4.8 RESULTS
# =============================================================================

print()
print("=" * 80)
print("EXTRACTING OPENVINO DEPLOYMENT RESULTS")
print("=" * 80)


def find_device_row(df, device_keyword):

    if "device" not in df.columns:
        raise ValueError(
            "Stage 4.8 comparison CSV does not "
            "contain a 'device' column."
        )

    mask = (
        df["device"]
        .astype(str)
        .str.lower()
        .str.contains(
            device_keyword.lower()
        )
    )

    matches = df[mask]

    if len(matches) == 0:
        return None

    return matches.iloc[0]


openvino_cpu = find_device_row(
    openvino_comparison_df,
    "CPU"
)

openvino_gpu = find_device_row(
    openvino_comparison_df,
    "GPU"
)


if openvino_cpu is None:

    raise RuntimeError(
        "OpenVINO CPU result not found."
    )


# =============================================================================
# 13. CONVERSION VALIDATION SUMMARY
# =============================================================================

print()
print("=" * 80)
print("CONVERSION FIDELITY")
print("=" * 80)

conversion_summary = {}

for column in conversion_df.columns:

    if len(conversion_df) > 0:

        conversion_summary[column] = (
            conversion_df.iloc[0][column]
        )


print(
    json.dumps(
        conversion_summary,
        indent=2,
        default=str
    )
)


# =============================================================================
# 14. DETERMINE CONVERSION PASS
# =============================================================================

def value_from_conversion(
    possible_names,
    default=None
):

    for name in possible_names:

        if name in conversion_df.columns:

            return conversion_df.iloc[0][name]

    return default


top1_agreement = value_from_conversion(
    [
        "top1_agreement",
        "Top-1 agreement",
        "top1_match",
        "Top1"
    ]
)

top5_overlap = value_from_conversion(
    [
        "top5_overlap",
        "Top-5 overlap",
        "top5_match"
    ]
)

cosine_similarity = value_from_conversion(
    [
        "cosine_similarity",
        "Output cosine similarity",
        "cosine"
    ]
)

max_abs_error = value_from_conversion(
    [
        "max_absolute_error",
        "Maximum absolute error",
        "max_abs_error"
    ]
)


# =============================================================================
# 15. FINAL DEPLOYMENT METRICS
# =============================================================================

print()
print("=" * 80)
print("FINAL DEPLOYMENT METRICS")
print("=" * 80)


def get_value(row, names, default=None):

    for name in names:

        if name in row.index:

            value = row[name]

            try:
                return float(value)
            except Exception:
                return value

    return default


# Retrieval
retrieval_top1 = get_value(
    mobile_retrieval,
    [
        "Top-1",
        "top1",
        "top1_accuracy",
        "Top1"
    ]
)

retrieval_top5 = get_value(
    mobile_retrieval,
    [
        "Top-5",
        "top5",
        "Top5"
    ]
)

retrieval_recall10 = get_value(
    mobile_retrieval,
    [
        "Recall@10",
        "recall@10",
        "Recall_10"
    ]
)

retrieval_mrr = get_value(
    mobile_retrieval,
    [
        "MRR",
        "mrr"
    ]
)

retrieval_mean_rank = get_value(
    mobile_retrieval,
    [
        "Mean Rank",
        "mean_rank"
    ]
)

retrieval_median_rank = get_value(
    mobile_retrieval,
    [
        "Median Rank",
        "median_rank"
    ]
)

# Computational
params = get_value(
    mobile_compute,
    [
        "params",
        "parameters",
        "Params"
    ]
)

model_size_mb = get_value(
    mobile_compute,
    [
        "size_mb",
        "model_size_mb",
        "Size_MB",
        "size"
    ]
)

gflops = get_value(
    mobile_compute,
    [
        "gflops",
        "flops_g",
        "GFLOPs"
    ]
)

pytorch_cpu_latency = float(
    openvino_cpu[
        "pytorch_latency_ms"
    ]
)

openvino_cpu_latency = float(
    openvino_cpu[
        "openvino_latency_ms"
    ]
)

cpu_speedup = float(
    openvino_cpu[
        "latency_speedup_x"
    ]
)

cpu_latency_reduction = float(
    openvino_cpu[
        "latency_reduction_percent"
    ]
)

pytorch_cpu_fps = float(
    openvino_cpu[
        "pytorch_fps"
    ]
)

openvino_cpu_fps = float(
    openvino_cpu[
        "openvino_fps"
    ]
)


if openvino_gpu is not None:

    pytorch_gpu_latency = float(
        openvino_gpu[
            "pytorch_latency_ms"
        ]
    )

    openvino_gpu_latency = float(
        openvino_gpu[
            "openvino_latency_ms"
        ]
    )

    gpu_speedup = float(
        openvino_gpu[
            "latency_speedup_x"
        ]
    )

    gpu_latency_reduction = float(
        openvino_gpu[
            "latency_reduction_percent"
        ]
    )

    pytorch_gpu_fps = float(
        openvino_gpu[
            "pytorch_fps"
        ]
    )

    openvino_gpu_fps = float(
        openvino_gpu[
            "openvino_fps"
        ]
    )

else:

    pytorch_gpu_latency = None
    openvino_gpu_latency = None
    gpu_speedup = None
    gpu_latency_reduction = None
    pytorch_gpu_fps = None
    openvino_gpu_fps = None


# =============================================================================
# 16. FINAL EVIDENCE TABLE
# =============================================================================

final_metrics = {

    "model":
        "MobileNetV3-Large",

    "precision":
        "FP32",

    "input_shape":
        "[1,3,224,224]",

    "output_classes":
        4764,

    # Retrieval
    "official_top1_percent":
        retrieval_top1,

    "official_top5_percent":
        retrieval_top5,

    "official_recall_at_10_percent":
        retrieval_recall10,

    "official_mrr":
        retrieval_mrr,

    "official_mean_rank":
        retrieval_mean_rank,

    "official_median_rank":
        retrieval_median_rank,

    # Computational
    "parameters":
        params,

    "model_size_mb":
        model_size_mb,

    "gflops":
        gflops,

    # OpenVINO conversion
    "conversion_top1_agreement":
        top1_agreement,

    "conversion_top5_overlap":
        top5_overlap,

    "conversion_cosine_similarity":
        cosine_similarity,

    "conversion_max_absolute_error":
        max_abs_error,

    # CPU
    "pytorch_cpu_latency_ms":
        pytorch_cpu_latency,

    "openvino_cpu_latency_ms":
        openvino_cpu_latency,

    "cpu_latency_speedup_x":
        cpu_speedup,

    "cpu_latency_reduction_percent":
        cpu_latency_reduction,

    "pytorch_cpu_fps":
        pytorch_cpu_fps,

    "openvino_cpu_fps":
        openvino_cpu_fps,

    # GPU
    "pytorch_gpu_latency_ms":
        pytorch_gpu_latency,

    "openvino_gpu_latency_ms":
        openvino_gpu_latency,

    "gpu_latency_speedup_x":
        gpu_speedup,

    "gpu_latency_reduction_percent":
        gpu_latency_reduction,

    "pytorch_gpu_fps":
        pytorch_gpu_fps,

    "openvino_gpu_fps":
        openvino_gpu_fps
}


final_metrics_df = pd.DataFrame(
    [final_metrics]
)


# =============================================================================
# 17. PRINT FINAL TABLE
# =============================================================================

print()
print(
    final_metrics_df.T.to_string(
        header=False
    )
)


# =============================================================================
# 18. DEPLOYMENT READINESS CHECKS
# =============================================================================

print()
print("=" * 80)
print("DEPLOYMENT READINESS CHECKS")
print("=" * 80)


checks = {}


# -------------------------------------------------------------------------
# IR exists
# -------------------------------------------------------------------------

checks["OpenVINO XML exists"] = (
    STAGE_4_7_XML.exists()
)

checks["OpenVINO BIN exists"] = (
    STAGE_4_7_BIN.exists()
)


# -------------------------------------------------------------------------
# Conversion fidelity
# -------------------------------------------------------------------------

try:

    cosine_numeric = float(
        cosine_similarity
    )

    checks["Conversion cosine >= 0.9999"] = (
        cosine_numeric >= 0.9999
    )

except Exception:

    checks["Conversion cosine >= 0.9999"] = False


try:

    max_error_numeric = float(
        max_abs_error
    )

    checks["Conversion max error <= 1e-3"] = (
        max_error_numeric <= 1e-3
    )

except Exception:

    checks["Conversion max error <= 1e-3"] = False


# -------------------------------------------------------------------------
# CPU benchmark
# -------------------------------------------------------------------------

checks["OpenVINO CPU benchmark available"] = (
    openvino_cpu is not None
)

checks["OpenVINO CPU FPS > 0"] = (
    openvino_cpu_fps > 0
)


# -------------------------------------------------------------------------
# Retrieval result
# -------------------------------------------------------------------------

checks["Official retrieval result available"] = (
    retrieval_top1 is not None
)


# -------------------------------------------------------------------------
# Overall
# -------------------------------------------------------------------------

overall_ready = all(
    checks.values()
)


for name, result in checks.items():

    print(
        f"{name:<45} : "
        f"{'PASS' if result else 'FAIL'}"
    )


print()
print(
    "OVERALL DEPLOYMENT READINESS:"
)

print(
    "  >>> "
    + (
        "PASS"
        if overall_ready
        else "CHECK FAILED"
    )
)


# =============================================================================
# 19. CREATE FINAL STAGE SUMMARY
# =============================================================================

summary = {

    "project":
        "CISLR",

    "stage":
        "4.9",

    "stage_name":
        "Final OpenVINO Deployment Evaluation",

    "status":
        "COMPLETE",

    "selected_model":
        "MobileNetV3-Large",

    "selection_basis":
        {
            "official_retrieval":
                "Best Top-1, Top-5, Recall@10 and MRR",

            "computational_efficiency":
                "Lowest GFLOPs and highest CPU FPS among candidates",

            "openvino_conversion":
                "Validated with matching Top-1/Top-5 and cosine similarity",

            "deployment_benchmark":
                "OpenVINO CPU and GPU benchmarks completed"
        },

    "official_cislr_retrieval": {

        "top1_percent":
            retrieval_top1,

        "top5_percent":
            retrieval_top5,

        "recall_at_10_percent":
            retrieval_recall10,

        "mrr":
            retrieval_mrr,

        "mean_rank":
            retrieval_mean_rank,

        "median_rank":
            retrieval_median_rank
    },

    "model_complexity": {

        "parameters":
            params,

        "model_size_mb":
            model_size_mb,

        "gflops":
            gflops
    },

    "openvino_conversion": {

        "precision":
            "FP32",

        "input_shape":
            "[1,3,224,224]",

        "output_shape":
            "[1,4764]",

        "xml":
            str(STAGE_4_7_XML),

        "bin":
            str(STAGE_4_7_BIN),

        "cosine_similarity":
            cosine_similarity,

        "max_absolute_error":
            max_abs_error,

        "top1_agreement":
            top1_agreement,

        "top5_overlap":
            top5_overlap
    },

    "openvino_cpu": {

        "mean_latency_ms":
            openvino_cpu_latency,

        "fps":
            openvino_cpu_fps,

        "pytorch_reference_latency_ms":
            pytorch_cpu_latency,

        "pytorch_reference_fps":
            pytorch_cpu_fps,

        "latency_speedup_x":
            cpu_speedup,

        "latency_reduction_percent":
            cpu_latency_reduction
    },

    "openvino_gpu": {

        "available":
            openvino_gpu is not None,

        "mean_latency_ms":
            openvino_gpu_latency,

        "fps":
            openvino_gpu_fps,

        "pytorch_reference_latency_ms":
            pytorch_gpu_latency,

        "pytorch_reference_fps":
            pytorch_gpu_fps,

        "latency_speedup_x":
            gpu_speedup,

        "latency_reduction_percent":
            gpu_latency_reduction
    },

    "deployment_readiness":
        checks,

    "overall_deployment_ready":
        overall_ready
}


# =============================================================================
# 20. SAVE FINAL METRICS CSV
# =============================================================================

final_csv = (
    OUTPUT_DIR /
    "stage4_9_final_deployment_metrics.csv"
)

final_metrics_df.to_csv(
    final_csv,
    index=False
)

print()
print("Saved final metrics CSV:")
print(final_csv)


# =============================================================================
# 21. SAVE DEPLOYMENT SUMMARY JSON
# =============================================================================

summary_json = (
    OUTPUT_DIR /
    "stage4_9_final_deployment_summary.json"
)

with open(
    summary_json,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        default=str
    )

print()
print("Saved final deployment summary:")
print(summary_json)


# =============================================================================
# 22. SAVE CHECKLIST CSV
# =============================================================================

checklist_df = pd.DataFrame(
    [
        {
            "check": name,
            "status": (
                "PASS"
                if result
                else "FAIL"
            )
        }
        for name, result in checks.items()
    ]
)

checklist_csv = (
    OUTPUT_DIR /
    "stage4_9_deployment_readiness_checklist.csv"
)

checklist_df.to_csv(
    checklist_csv,
    index=False
)

print()
print("Saved readiness checklist:")
print(checklist_csv)


# =============================================================================
# 23. PLOT 1 — RETRIEVAL PERFORMANCE
# =============================================================================

print()
print("=" * 80)
print("CREATING FINAL PLOTS")
print("=" * 80)


retrieval_plot_path = (
    PLOTS_DIR /
    "stage4_9_retrieval_performance.png"
)


retrieval_values = [
    retrieval_top1,
    retrieval_top5,
    retrieval_recall10
]

retrieval_labels = [
    "Top-1",
    "Top-5",
    "Recall@10"
]

plt.figure(
    figsize=(8, 5)
)

bars = plt.bar(
    retrieval_labels,
    retrieval_values
)

plt.ylabel(
    "Percentage (%)"
)

plt.title(
    "MobileNetV3-Large — Official CISLR Retrieval Performance"
)

plt.ylim(
    0,
    max(retrieval_values) * 1.25
)

for bar, value in zip(
    bars,
    retrieval_values
):

    plt.text(
        bar.get_x() +
        bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

plt.savefig(
    retrieval_plot_path,
    dpi=300
)

plt.close()

print(
    "Saved:",
    retrieval_plot_path
)


# =============================================================================
# 24. PLOT 2 — PYTORCH VS OPENVINO CPU
# =============================================================================

cpu_plot_path = (
    PLOTS_DIR /
    "stage4_9_cpu_pytorch_vs_openvino.png"
)

cpu_labels = [
    "PyTorch CPU",
    "OpenVINO CPU"
]

cpu_latencies = [
    pytorch_cpu_latency,
    openvino_cpu_latency
]

plt.figure(
    figsize=(8, 5)
)

bars = plt.bar(
    cpu_labels,
    cpu_latencies
)

plt.ylabel(
    "Mean Latency (ms)"
)

plt.title(
    "CPU Inference Latency — PyTorch vs OpenVINO"
)

for bar, value in zip(
    bars,
    cpu_latencies
):

    plt.text(
        bar.get_x() +
        bar.get_width() / 2,
        bar.get_height(),
        f"{value:.3f} ms",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

plt.savefig(
    cpu_plot_path,
    dpi=300
)

plt.close()

print(
    "Saved:",
    cpu_plot_path
)


# =============================================================================
# 25. PLOT 3 — FPS COMPARISON
# =============================================================================

fps_plot_path = (
    PLOTS_DIR /
    "stage4_9_fps_comparison.png"
)

fps_labels = [
    "PyTorch CPU",
    "OpenVINO CPU"
]

fps_values = [
    pytorch_cpu_fps,
    openvino_cpu_fps
]

if openvino_gpu is not None:

    fps_labels.extend(
        [
            "PyTorch GPU",
            "OpenVINO GPU"
        ]
    )

    fps_values.extend(
        [
            pytorch_gpu_fps,
            openvino_gpu_fps
        ]
    )


plt.figure(
    figsize=(9, 5)
)

bars = plt.bar(
    fps_labels,
    fps_values
)

plt.ylabel(
    "FPS"
)

plt.title(
    "Inference Throughput — PyTorch vs OpenVINO"
)

for bar, value in zip(
    bars,
    fps_values
):

    plt.text(
        bar.get_x() +
        bar.get_width() / 2,
        bar.get_height(),
        f"{value:.2f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

plt.savefig(
    fps_plot_path,
    dpi=300
)

plt.close()

print(
    "Saved:",
    fps_plot_path
)


# =============================================================================
# 26. PLOT 4 — OPENVINO LATENCY DISTRIBUTION
# =============================================================================

latency_distribution_path = (
    PLOTS_DIR /
    "stage4_9_openvino_latency_distribution.png"
)


plt.figure(
    figsize=(9, 5)
)

devices = list(
    latency_df["device"]
    .dropna()
    .unique()
)

for device in devices:

    values = latency_df.loc[
        latency_df["device"] == device,
        "latency_ms"
    ].values

    if len(values) > 0:

        plt.hist(
            values,
            bins=30,
            alpha=0.6,
            label=str(device)
        )

plt.xlabel(
    "Latency (ms)"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "OpenVINO Inference Latency Distribution"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    latency_distribution_path,
    dpi=300
)

plt.close()

print(
    "Saved:",
    latency_distribution_path
)


# =============================================================================
# 27. FINAL ARTIFACT HASHES
# =============================================================================

print()
print("=" * 80)
print("GENERATING FINAL ARTIFACT HASHES")
print("=" * 80)


output_artifacts = {

    "final_metrics_csv":
        final_csv,

    "deployment_summary_json":
        summary_json,

    "readiness_checklist_csv":
        checklist_csv,

    "retrieval_plot":
        retrieval_plot_path,

    "cpu_comparison_plot":
        cpu_plot_path,

    "fps_comparison_plot":
        fps_plot_path,

    "latency_distribution_plot":
        latency_distribution_path
}


artifact_hashes = {}

for name, path in output_artifacts.items():

    artifact_hashes[name] = {
        "path": str(path),
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size
    }


# Also record the hashes of the critical input artifacts.

critical_inputs = {

    "stage4_7_xml":
        STAGE_4_7_XML,

    "stage4_7_bin":
        STAGE_4_7_BIN,

    "stage4_8_comparison":
        STAGE_4_8_COMPARISON,

    "stage4_8_benchmark":
        STAGE_4_8_BENCHMARK,

    "stage4_5_retrieval":
        STAGE_4_5_COMPARISON,

    "stage4_3_compute":
        STAGE_4_3_CSV
}


critical_input_hashes = {}

for name, path in critical_inputs.items():

    critical_input_hashes[name] = {
        "path": str(path),
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size
    }


hash_report = {

    "stage":
        "4.9",

    "output_artifacts":
        artifact_hashes,

    "critical_input_artifacts":
        critical_input_hashes
}


hash_json = (
    OUTPUT_DIR /
    "stage4_9_final_artifact_hashes.json"
)

with open(
    hash_json,
    "w"
) as f:

    json.dump(
        hash_report,
        f,
        indent=2
    )

print()
print("Saved final hashes:")
print(hash_json)


# =============================================================================
# 28. CREATE FINAL HUMAN-READABLE REPORT
# =============================================================================

report_path = (
    OUTPUT_DIR /
    "stage4_9_final_report.txt"
)


report_lines = [

    "=" * 80,

    "CISLR — FINAL OPENVINO DEPLOYMENT EVALUATION",

    "=" * 80,

    "",

    "SELECTED MODEL",
    "MobileNetV3-Large",

    "",

    "DEPLOYMENT BACKEND",
    "OpenVINO FP32",

    "",

    "INPUT",
    "[1, 3, 224, 224]",

    "",

    "OUTPUT",
    "[1, 4764]",

    "",

    "=" * 80,

    "OFFICIAL CISLR RETRIEVAL",

    "=" * 80,

    f"Top-1       : {retrieval_top1:.2f}%",

    f"Top-5       : {retrieval_top5:.2f}%",

    f"Recall@10   : {retrieval_recall10:.2f}%",

    f"MRR         : {retrieval_mrr:.4f}",

    f"Mean Rank   : {retrieval_mean_rank:.2f}",

    f"Median Rank : {retrieval_median_rank:.2f}",

    "",

    "=" * 80,

    "MODEL COMPLEXITY",

    "=" * 80,

    f"Parameters  : {params}",

    f"Model size  : {model_size_mb:.3f} MB",

    f"GFLOPs      : {gflops:.3f}",

    "",

    "=" * 80,

    "OPENVINO CONVERSION",

    "=" * 80,

    "Precision   : FP32",

    f"XML         : {STAGE_4_7_XML}",

    f"BIN         : {STAGE_4_7_BIN}",

    f"Cosine      : {cosine_similarity}",

    f"Max error   : {max_abs_error}",

    f"Top-1       : {top1_agreement}",

    f"Top-5       : {top5_overlap}",

    "",

    "=" * 80,

    "OPENVINO CPU DEPLOYMENT",

    "=" * 80,

    f"PyTorch latency      : {pytorch_cpu_latency:.3f} ms",

    f"OpenVINO latency     : {openvino_cpu_latency:.3f} ms",

    f"Latency speedup      : {cpu_speedup:.3f}x",

    f"Latency reduction    : {cpu_latency_reduction:.2f}%",

    f"PyTorch FPS          : {pytorch_cpu_fps:.2f}",

    f"OpenVINO FPS         : {openvino_cpu_fps:.2f}",

    "",

    "=" * 80,

    "OPENVINO GPU DEPLOYMENT",

    "=" * 80,

]

if openvino_gpu is not None:

    report_lines.extend(
        [
            f"PyTorch latency      : {pytorch_gpu_latency:.3f} ms",
            f"OpenVINO latency     : {openvino_gpu_latency:.3f} ms",
            f"Latency speedup      : {gpu_speedup:.3f}x",
            f"Latency reduction    : {gpu_latency_reduction:.2f}%",
            f"PyTorch FPS          : {pytorch_gpu_fps:.2f}",
            f"OpenVINO FPS         : {openvino_gpu_fps:.2f}",
        ]
    )

else:

    report_lines.append(
        "OpenVINO GPU benchmark was not available."
    )


report_lines.extend(
    [
        "",

        "=" * 80,

        "DEPLOYMENT READINESS",

        "=" * 80,

    ]
)


for name, result in checks.items():

    report_lines.append(
        f"{name}: "
        f"{'PASS' if result else 'FAIL'}"
    )


report_lines.extend(
    [

        "",

        "OVERALL:",
        "PASS" if overall_ready else "CHECK FAILED",

        "",

        "=" * 80,

        "PIPELINE STATUS",

        "=" * 80,

        "4.1  Dataset / Evaluation Constraints       COMPLETE",

        "4.2  Canonical Training Protocol             COMPLETE",

        "4.3  Computational Benchmark                 COMPLETE",

        "4.4  Model Training                          COMPLETE",

        "4.5  Official CISLR Evaluation               COMPLETE",

        "4.6  Model Comparison & Selection            COMPLETE",

        "4.7  OpenVINO Conversion                     COMPLETE",

        "4.8  OpenVINO Benchmark                      COMPLETE",

        "4.9  Final Deployment Evaluation             COMPLETE",

        "",

        "FINAL SELECTED MODEL:",
        "MobileNetV3-Large",

        "",

        "FINAL DEPLOYMENT BACKEND:",
        "OpenVINO FP32",

        "",

        "NO FURTHER MODEL TRAINING IS REQUIRED FOR THIS PIPELINE.",

        "=" * 80
    ]
)


with open(
    report_path,
    "w"
) as f:

    f.write(
        "\n".join(
            report_lines
        )
    )


print()
print("Saved final human-readable report:")
print(report_path)


# =============================================================================
# 29. FINAL CONSOLE SUMMARY
# =============================================================================

print()
print("=" * 80)
print("STAGE 4.9 COMPLETE")
print("=" * 80)

print()
print("FINAL SELECTED MODEL:")
print("  >>> MobileNetV3-Large")

print()
print("FINAL DEPLOYMENT BACKEND:")
print("  >>> OpenVINO FP32")

print()
print("OFFICIAL CISLR RETRIEVAL:")
print(
    f"  Top-1      : {retrieval_top1:.2f}%"
)

print(
    f"  Top-5      : {retrieval_top5:.2f}%"
)

print(
    f"  Recall@10  : {retrieval_recall10:.2f}%"
)

print(
    f"  MRR        : {retrieval_mrr:.4f}"
)

print()
print("OPENVINO CPU:")
print(
    f"  Latency    : {openvino_cpu_latency:.3f} ms"
)

print(
    f"  FPS        : {openvino_cpu_fps:.2f}"
)

print(
    f"  Speedup    : {cpu_speedup:.3f}x"
)

print(
    f"  Reduction  : {cpu_latency_reduction:.2f}%"
)


if openvino_gpu is not None:

    print()
    print("OPENVINO GPU:")
    print(
        f"  Latency    : {openvino_gpu_latency:.3f} ms"
    )

    print(
        f"  FPS        : {openvino_gpu_fps:.2f}"
    )

    print(
        f"  Speedup    : {gpu_speedup:.3f}x"
    )

    print(
        f"  Reduction  : {gpu_latency_reduction:.2f}%"
    )


print()
print("CONVERSION FIDELITY:")

print(
    f"  Cosine     : {cosine_similarity}"
)

print(
    f"  Max error  : {max_abs_error}"
)

print(
    f"  Top-1      : {top1_agreement}"
)

print(
    f"  Top-5      : {top5_overlap}"
)

print()
print("DEPLOYMENT READINESS:")
print(
    "  >>> "
    + (
        "PASS"
        if overall_ready
        else "CHECK FAILED"
    )
)

print()
print("FINAL OUTPUT DIRECTORY:")
print(OUTPUT_DIR)

print()
print("FINAL ARTIFACTS:")

print(
    "  ",
    final_csv
)

print(
    "  ",
    summary_json
)

print(
    "  ",
    checklist_csv
)

print(
    "  ",
    hash_json
)

print(
    "  ",
    report_path
)

print()
print("PLOTS:")

print(
    "  ",
    retrieval_plot_path
)

print(
    "  ",
    cpu_plot_path
)

print(
    "  ",
    fps_plot_path
)

print(
    "  ",
    latency_distribution_path
)

print()
print("=" * 80)
print("CISLR OPENVINO PIPELINE — FINAL STAGE COMPLETE")
print("=" * 80)

print()
print("STAGES 4.1 THROUGH 4.9 ARE COMPLETE.")
print()
print("DO NOT RERUN:")
print("  4.3 Computational Benchmark")
print("  4.4 Model Training")
print("  4.5 Official CISLR Evaluation")
print("  4.6 Model Comparison & Selection")
print("  4.7 OpenVINO Conversion")
print("  4.8 OpenVINO Benchmark")

print()
print(">>> CISLR MODEL SELECTION + OPENVINO DEPLOYMENT PIPELINE COMPLETE <<<")
print("=" * 80)

CISLR — STAGE 4.9 FINAL OPENVINO DEPLOYMENT EVALUATION

Project root:
/home/dipshikha/Music/cao/CISLR_RESEARCH

Final selected model:
MobileNetV3-Large

Final deployment backend:
OpenVINO FP32

Output:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_9_final_deployment_evaluation

ENVIRONMENT
Python       : 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
Platform     : Linux-7.0.0-31-generic-x86_64-with-glibc2.39
Architecture : x86_64

VERIFYING PREVIOUS STAGE ARTIFACTS
Stage 4.3 computational CSV              : FOUND
Stage 4.3 computational JSON             : FOUND
Stage 4.5 retrieval comparison           : FOUND
Stage 4.5 retrieval comparison JSON      : FOUND
Stage 4.7 OpenVINO XML                   : FOUND
Stage 4.7 OpenVINO BIN                   : FOUND
Stage 4.7 validation CSV                 : FOUND
Stage 4.7 metadata                       : FOUND
Stage 4.8 comparison CSV                 : FOUND
Stage 4.8 benchmark JSON                 : FOUND
Stage 4.8 latency CSV      

In [1]:
# =============================================================================
# CISLR — STAGE 9.2
# REAL-DATA OPENVINO EMBEDDING VALIDATION
# =============================================================================
#
# GOAL
# ----
# Verify that the new Stage 9.1 OpenVINO 960-D model reproduces the
# EXISTING Stage 4.5 MobileNetV3-Large embedding space on REAL CISLR videos.
#
# NO TRAINING.
# NO FILE MODIFICATION.
# NO TEST-SET TUNING.
#
# Validation:
#
# real CISLR video
#      ↓
# exact Stage 4.5 sampled frames
#      ↓
# exact ImageNet preprocessing
#      ↓
# Stage 9.1 OpenVINO 960-D model
#      ↓
# mean over 8 frame embeddings
#      ↓
# L2 normalize
#      ↓
# compare against stored Stage 4.5 960-D video embedding
#
# =============================================================================

from pathlib import Path
import ast
import json

import cv2
import numpy as np
import pandas as pd
import openvino as ov


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path("/home/dipshikha/Music/cao/CISLR_RESEARCH")

OV_XML = (
    ROOT
    / "audit/stage9_final_openvino/embedding_fp32/"
      "mobilenet_v3_large_embedding_960_fp32_static.xml"
)

STAGE45 = (
    ROOT
    / "audit/stage4_5_official_evaluation/embeddings"
)

PROTOTYPE_META_PATH = (
    STAGE45
    / "mobilenetv3_large_prototype_metadata.csv"
)

PROTOTYPE_EMBED_PATH = (
    STAGE45
    / "mobilenetv3_large_prototype_embeddings.npy"
)

TEST_META_PATH = (
    STAGE45
    / "mobilenetv3_large_test_metadata.csv"
)

TEST_EMBED_PATH = (
    STAGE45
    / "mobilenetv3_large_test_embeddings.npy"
)

OUTPUT_DIR = (
    ROOT
    / "audit/stage9_final_openvino/"
      "real_data_validation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_CSV = (
    OUTPUT_DIR
    / "stage9_2_real_data_validation.csv"
)

SUMMARY_JSON = (
    OUTPUT_DIR
    / "stage9_2_real_data_validation_summary.json"
)


# =============================================================================
# 2. START
# =============================================================================

print("=" * 100)
print("CISLR — STAGE 9.2")
print("REAL-DATA OPENVINO EMBEDDING VALIDATION")
print("=" * 100)


for name, path in [
    ("OpenVINO model", OV_XML),
    ("Prototype metadata", PROTOTYPE_META_PATH),
    ("Prototype embeddings", PROTOTYPE_EMBED_PATH),
    ("Test metadata", TEST_META_PATH),
    ("Test embeddings", TEST_EMBED_PATH),
]:

    print(f"\n{name}:")
    print(path)
    print("Exists:", path.exists())

    assert path.exists(), path


# =============================================================================
# 3. LOAD FROZEN STAGE 4.5 ARTIFACTS
# =============================================================================

prototype_meta = pd.read_csv(
    PROTOTYPE_META_PATH
)

test_meta = pd.read_csv(
    TEST_META_PATH
)

prototype_reference = np.load(
    PROTOTYPE_EMBED_PATH,
    allow_pickle=False
)

test_reference = np.load(
    TEST_EMBED_PATH,
    allow_pickle=False
)


print("\n" + "-" * 100)
print("STAGE 4.5 REFERENCE ARTIFACTS")
print("-" * 100)

print(
    "Prototype metadata :",
    prototype_meta.shape
)

print(
    "Prototype embedding:",
    prototype_reference.shape
)

print(
    "Test metadata      :",
    test_meta.shape
)

print(
    "Test embedding     :",
    test_reference.shape
)


assert len(prototype_meta) == len(prototype_reference)
assert len(test_meta) == len(test_reference)

assert prototype_reference.shape[1] == 960
assert test_reference.shape[1] == 960

print("\n[OK] Stage 4.5 reference artifacts are aligned.")


# =============================================================================
# 4. LOAD OPENVINO MODEL
# =============================================================================

core = ov.Core()

model = core.read_model(
    str(OV_XML)
)

compiled = core.compile_model(
    model,
    "CPU"
)

input_port = compiled.input(0)
output_port = compiled.output(0)


print("\n" + "-" * 100)
print("OPENVINO MODEL")
print("-" * 100)

print(
    "Input :",
    tuple(input_port.shape)
)

print(
    "Output:",
    tuple(output_port.shape)
)


assert tuple(input_port.shape) == (
    1, 3, 224, 224
)

assert tuple(output_port.shape) == (
    1, 960
)

print("\n[OK] OpenVINO model ready.")


# =============================================================================
# 5. EXACT MOBILENET IMAGENET PREPROCESSING
# =============================================================================
#
# torchvision MobileNetV3 ImageNet preprocessing:
#
# RGB
# resize / crop equivalent for square CISLR frames
# float [0,1]
# ImageNet normalization
#
# Stage 4.x uses 224 × 224 inputs.
#
# =============================================================================

IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
).reshape(1, 1, 3)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
).reshape(1, 1, 3)


def preprocess_frame(frame_bgr):

    # BGR -> RGB
    frame = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    # CISLR source frames are square.
    # Resize directly to trained input resolution.
    frame = cv2.resize(
        frame,
        (224, 224),
        interpolation=cv2.INTER_LINEAR
    )

    frame = (
        frame.astype(np.float32)
        / 255.0
    )

    frame = (
        frame - IMAGENET_MEAN
    ) / IMAGENET_STD

    # HWC -> CHW
    frame = np.transpose(
        frame,
        (2, 0, 1)
    )

    # Add batch dimension
    frame = np.expand_dims(
        frame,
        axis=0
    )

    return np.ascontiguousarray(
        frame,
        dtype=np.float32
    )


# =============================================================================
# 6. PARSE EXACT STAGE 4.5 FRAME INDICES
# =============================================================================

def parse_indices(value):

    if isinstance(value, str):

        indices = ast.literal_eval(
            value
        )

    else:

        indices = list(value)

    return [
        int(x)
        for x in indices
    ]


# =============================================================================
# 7. READ EXACT VIDEO FRAMES
# =============================================================================

def read_selected_frames(
    video_path,
    indices
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open video: "
            f"{video_path}"
        )

    frames = []

    for frame_index in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_index)
        )

        ok, frame = cap.read()

        if not ok or frame is None:

            cap.release()

            raise RuntimeError(
                f"Could not read frame "
                f"{frame_index} from "
                f"{video_path}"
            )

        frames.append(frame)

    cap.release()

    return frames


# =============================================================================
# 8. OPENVINO VIDEO EMBEDDING
# =============================================================================

def openvino_video_embedding(
    video_path,
    sample_indices
):

    frames = read_selected_frames(
        video_path,
        sample_indices
    )

    frame_embeddings = []

    for frame in frames:

        x = preprocess_frame(
            frame
        )

        result = compiled([x])

        embedding = np.asarray(
            result[output_port],
            dtype=np.float32
        )[0]

        assert embedding.shape == (
            960,
        )

        frame_embeddings.append(
            embedding
        )

    frame_embeddings = np.stack(
        frame_embeddings,
        axis=0
    )

    # Exact final temporal aggregation
    video_embedding = np.mean(
        frame_embeddings,
        axis=0
    )

    norm = np.linalg.norm(
        video_embedding
    )

    if norm <= 1e-12:

        raise RuntimeError(
            "Zero-norm video embedding."
        )

    video_embedding = (
        video_embedding / norm
    )

    return (
        video_embedding.astype(
            np.float32
        ),
        frame_embeddings
    )


# =============================================================================
# 9. COSINE FUNCTION
# =============================================================================

def cosine(a, b):

    a = np.asarray(
        a,
        dtype=np.float64
    ).reshape(-1)

    b = np.asarray(
        b,
        dtype=np.float64
    ).reshape(-1)

    denom = (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )

    if denom <= 1e-15:
        return np.nan

    return float(
        np.dot(a, b)
        / denom
    )


# =============================================================================
# 10. SELECT VALIDATION VIDEOS
# =============================================================================
#
# We validate BOTH:
#
#   5 prototype videos
#   5 test videos
#
# Deterministic selection.
#
# First rows are intentionally useful because the precheck already
# confirmed their metadata and paths.
#
# =============================================================================

prototype_selection = (
    prototype_meta
    .query("status == 'OK'")
    .head(5)
    .copy()
)

test_selection = (
    test_meta
    .query("status == 'OK'")
    .head(5)
    .copy()
)


print("\n" + "-" * 100)
print("VALIDATION SET")
print("-" * 100)

print("\nPrototype videos:")

for idx, row in prototype_selection.iterrows():

    print(
        row["uid"],
        "|",
        row["gloss"],
        "|",
        row["video_path"]
    )


print("\nTest videos:")

for idx, row in test_selection.iterrows():

    print(
        row["uid"],
        "|",
        row["gloss"],
        "|",
        row["video_path"]
    )


# =============================================================================
# 11. VALIDATE
# =============================================================================

records = []


def validate_group(
    metadata,
    reference_embeddings,
    selected_rows,
    split_name
):

    for dataframe_index, row in selected_rows.iterrows():

        uid = str(
            row["uid"]
        )

        gloss = str(
            row["gloss"]
        )

        video_path = Path(
            row["video_path"]
        )

        indices = parse_indices(
            row["sample_indices"]
        )

        print(
            "\n" +
            "=" * 100
        )

        print(
            f"{split_name.upper()} | "
            f"{uid} | {gloss}"
        )

        print(
            "Video:",
            video_path
        )

        print(
            "Frames:",
            indices
        )

        if not video_path.exists():

            print(
                "[FAIL] Video file does not exist."
            )

            records.append({
                "split": split_name,
                "uid": uid,
                "gloss": gloss,
                "video_path": str(video_path),
                "sample_indices": str(indices),
                "status": "VIDEO_MISSING",
                "cosine_vs_stage45": np.nan,
                "max_abs_error": np.nan,
                "mean_abs_error": np.nan,
            })

            continue


        # -------------------------------------------------------------
        # Important:
        #
        # dataframe_index corresponds to embedding row because Stage 4.5
        # metadata and .npy arrays were saved in aligned order.
        # -------------------------------------------------------------

        reference = np.asarray(
            reference_embeddings[
                dataframe_index
            ],
            dtype=np.float32
        )

        reference_norm = float(
            np.linalg.norm(reference)
        )


        ov_embedding, frame_embeddings = (
            openvino_video_embedding(
                video_path,
                indices
            )
        )


        similarity = cosine(
            ov_embedding,
            reference
        )


        # Normalize stored reference before direct numerical comparison.
        reference_normalized = (
            reference
            /
            (
                np.linalg.norm(reference)
                + 1e-12
            )
        )


        error = np.abs(
            ov_embedding
            - reference_normalized
        )


        max_error = float(
            np.max(error)
        )

        mean_error = float(
            np.mean(error)
        )


        print(
            "Frame embedding shape :",
            frame_embeddings.shape
        )

        print(
            "OpenVINO video shape   :",
            ov_embedding.shape
        )

        print(
            "Reference shape        :",
            reference.shape
        )

        print(
            "Reference norm         :",
            reference_norm
        )

        print(
            "OpenVINO norm          :",
            float(
                np.linalg.norm(
                    ov_embedding
                )
            )
        )

        print(
            "Cosine vs Stage 4.5    :",
            f"{similarity:.10f}"
        )

        print(
            "Max abs error          :",
            f"{max_error:.10e}"
        )

        print(
            "Mean abs error         :",
            f"{mean_error:.10e}"
        )


        records.append({

            "split":
                split_name,

            "uid":
                uid,

            "gloss":
                gloss,

            "video_path":
                str(video_path),

            "sample_indices":
                str(indices),

            "status":
                "OK",

            "reference_norm":
                reference_norm,

            "openvino_norm":
                float(
                    np.linalg.norm(
                        ov_embedding
                    )
                ),

            "cosine_vs_stage45":
                similarity,

            "max_abs_error":
                max_error,

            "mean_abs_error":
                mean_error,
        })


validate_group(
    prototype_meta,
    prototype_reference,
    prototype_selection,
    "prototype"
)

validate_group(
    test_meta,
    test_reference,
    test_selection,
    "test"
)


# =============================================================================
# 12. RESULTS
# =============================================================================

results = pd.DataFrame(
    records
)

print("\n" + "=" * 100)
print("STAGE 9.2 RESULTS")
print("=" * 100)

display_columns = [
    "split",
    "uid",
    "gloss",
    "status",
    "cosine_vs_stage45",
    "max_abs_error",
    "mean_abs_error",
]

print(
    results[
        display_columns
    ].to_string(
        index=False
    )
)


# =============================================================================
# 13. SUMMARY
# =============================================================================

valid = results[
    results["status"] == "OK"
].copy()


if len(valid) == 0:

    raise RuntimeError(
        "No videos were successfully validated."
    )


mean_cosine = float(
    valid[
        "cosine_vs_stage45"
    ].mean()
)

min_cosine = float(
    valid[
        "cosine_vs_stage45"
    ].min()
)

median_cosine = float(
    valid[
        "cosine_vs_stage45"
    ].median()
)

mean_max_error = float(
    valid[
        "max_abs_error"
    ].mean()
)


print("\n" + "-" * 100)
print("SUMMARY")
print("-" * 100)

print(
    "Videos requested :",
    len(results)
)

print(
    "Videos validated :",
    len(valid)
)

print(
    "Mean cosine      :",
    f"{mean_cosine:.10f}"
)

print(
    "Median cosine    :",
    f"{median_cosine:.10f}"
)

print(
    "Minimum cosine   :",
    f"{min_cosine:.10f}"
)

print(
    "Mean max error   :",
    f"{mean_max_error:.10e}"
)


# =============================================================================
# 14. CONSERVATIVE PASS CRITERION
# =============================================================================
#
# Do NOT demand bitwise equality because:
# - OpenVINO and PyTorch kernels differ slightly.
# - decoding/preprocessing implementation can introduce tiny differences.
#
# But embeddings should be extremely close if we reconstructed the
# Stage 4.5 pipeline correctly.
#
# =============================================================================

PASS_COSINE = 0.999

all_embedding_matches = bool(
    (
        valid["cosine_vs_stage45"]
        >= PASS_COSINE
    ).all()
)


print("\nPass threshold:")
print(
    f"cosine >= {PASS_COSINE}"
)

print(
    "All embeddings pass:",
    all_embedding_matches
)


# =============================================================================
# 15. SAVE
# =============================================================================

results.to_csv(
    RESULT_CSV,
    index=False
)


summary = {

    "stage":
        "9.2",

    "purpose":
        "Real CISLR validation of Stage 9.1 OpenVINO 960-D embedding model",

    "videos_requested":
        int(len(results)),

    "videos_validated":
        int(len(valid)),

    "prototype_videos_requested":
        int(
            (
                results["split"]
                == "prototype"
            ).sum()
        ),

    "test_videos_requested":
        int(
            (
                results["split"]
                == "test"
            ).sum()
        ),

    "mean_cosine_vs_stage45":
        mean_cosine,

    "median_cosine_vs_stage45":
        median_cosine,

    "minimum_cosine_vs_stage45":
        min_cosine,

    "pass_cosine_threshold":
        PASS_COSINE,

    "all_embeddings_pass":
        all_embedding_matches,

    "training_performed":
        False,

    "test_set_tuning":
        False,

    "model_modified":
        False
}


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print("\nSaved:")
print(RESULT_CSV)
print(SUMMARY_JSON)


# =============================================================================
# 16. FINAL DECISION
# =============================================================================

print("\n" + "=" * 100)

if all_embedding_matches:

    print(
        "STAGE 9.2 — PASS"
    )

    print("=" * 100)

    print("""
The Stage 9.1 OpenVINO model reproduces the frozen Stage 4.5
MobileNetV3-Large 960-D embedding space on real CISLR videos.

The OpenVINO embedding extractor can therefore proceed to the
final retrieval-engine integration stage.
""")

else:

    print(
        "STAGE 9.2 — INVESTIGATION REQUIRED"
    )

    print("=" * 100)

    print("""
The OpenVINO model itself was already numerically validated in Stage 9.1,
but the real-video embeddings do not reproduce the stored Stage 4.5
embeddings closely enough.

DO NOT retrain anything.

If this occurs, the likely issue is historical preprocessing or frame
decoding differences. We will identify that difference before proceeding.
""")

CISLR — STAGE 9.2
REAL-DATA OPENVINO EMBEDDING VALIDATION

OpenVINO model:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage9_final_openvino/embedding_fp32/mobilenet_v3_large_embedding_960_fp32_static.xml
Exists: True

Prototype metadata:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_prototype_metadata.csv
Exists: True

Prototype embeddings:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_prototype_embeddings.npy
Exists: True

Test metadata:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_test_metadata.csv
Exists: True

Test embeddings:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_test_embeddings.npy
Exists: True

----------------------------------------------------------------------------------------------------
STAGE 4.5 REFERENCE ARTIFACTS
-----------